In [1]:
import os 
import certifi 
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests

In [2]:
from langchain.agents import create_react_agent, AgentExecutor

In [3]:
# load env variables
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [4]:
search_tool = TavilySearchResults(max_results=2)

In [5]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [6]:
result = search_tool.invoke("Give me a summary of today's news articles")
result

[{'url': 'https://www.youtube.com/watch?v=E1w4QUAXcwQ',
  'content': "[3:07] such energy attacks. Both sides denied Trump's claim, though Ukrainian President Vladimir Zalinski said he was\n[3:14] willing to halt attacks on energy infrastructure if Russia reciprocated.\n[3:20] Meanwhile, 11 sailors went missing in the Black Sea today after an apparent drone attack on two cargo ships off the coast of Bulgaria, one of which sank. It\n[3:29] was the second deadly attack of its kind in the past week and prompted Bulgaria to summon Russia's ambassador. This is the Bulgarian Prime Minister Rman Radv.\n[3:42] We do not possess solid clear evidence to level an accusation against any specific party. However, logic suggests\n[3:49] that the outcome works to the advantage of the Russian Federation. For this reason, I am summoning the Russian ambassador for a meeting today. [...] [9:12] My opponent is the only one on this stage who had the opportunity to vote to end the war, but three times in a ro

In [7]:
# LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    api_key=OPENAI_API_KEY
)

In [8]:
response = llm.invoke("Write a poem about AI Engineers")
response

AIMessage(content='In labs of steel and circuits bright,\nAI engineers work through the night,\nCreating minds of silicon and wire,\nTo think and learn and never tire.\n\nThey craft algorithms with precision,\nTo mimic human decision,\nThey teach machines to see and hear,\nTo navigate and persevere.\n\nWith code as their language, they build,\nIntelligence that is skilled,\nIn solving problems, big and small,\nThey push the boundaries, break down walls.\n\nTheir work is groundbreaking, innovative,\nTheir minds are sharp, their skills creative,\nAI engineers, the architects of tomorrow,\nShaping a future free of sorrow.\n\nSo let us praise their dedication,\nTheir passion for innovation,\nFor they are the ones who pave the way,\nFor a world where AI will reign one day.', response_metadata={'token_usage': {'completion_tokens': 151, 'prompt_tokens': 13, 'total_tokens': 164, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_p

In [9]:
# prompt
prompt = hub.pull("hwchase17/react")

c:\Users\midem\anaconda3\envs\langchain_ai\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [10]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [11]:
# tools
tools = [search_tool, get_weather_data]

In [12]:
# Create agent
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

In [13]:
# Create agent executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [14]:
response = agent_executor.invoke({
    "input": ("Where is Worksop in the UK. What is the current weather in Worksop, UK.")
})



> Entering new AgentExecutor chain...
I should first search for the location of Worksop in the UK and then get the current weather information for Worksop.
Action: tavily_search_results_json
Action Input: "Worksop location in the UK"[{'url': 'https://en.wikipedia.org/wiki/Worksop', 'content': 'Worksop is located in Nottinghamshire\nWorksop\n\nWorksop (/ˈwɜːrksɒp/ WURK-sop) is a market town in the Bassetlaw District in Nottinghamshire, England. Located close to Nottinghamshire\'s borders with South Yorkshire and Derbyshire, it is on the River Ryton and not far from the northern edge of Sherwood Forest. The population of the town was recorded at 44,733 in the 2021 Census. To the south of Worksop is the area of the Dukeries.\n\n## Toponymy [...] | Worksop | |\n --- |\n| Town | |\n| Bridge Street  Clumber Park  Mr Straw\'s House  Worksop Priory  Chesterfield Canal | |\n| Worksop is located in Nottinghamshire Worksop  Worksop  Location within Nottinghamshire | |\n| Population | 44,733  |\

In [15]:
print(response["output"])

Worksop is located in Nottinghamshire, England. The current weather in Worksop, UK is overcast with a temperature of 12°C and 64% humidity.
